<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/responsiveness_matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Matching visual responsiveness, not only tuning, across recording techniques

Neuropixels, mesoscope 2-photon and SLAP2 do not count the same cells as "visually responsive".
Neuropixels detects most visual units (≈87 % respond to the standard grating), calcium imaging about
half (mesoscope ≈49 %, SLAP2 ≈42 %), and the default rules differ: the Neuropixels rule admits cells
*suppressed* by the standard, the imaging rule requires a positive response. Because the detection
threshold pre-selects a tuning-biased subset, balancing orientation tuning alone is not enough.
This notebook tests three levels of matching on the feature-oddball index:

1. **Same rule everywhere** — apply the imaging (excitatory-only) responsiveness rule to Neuropixels.
2. **Detection-floor test** — keep only the most strongly driven Neuropixels cells, as imaging does implicitly.
3. **Joint balancing** — equalise the distribution of response strength *and* tuning preference, then
   compare OI; inference resamples **mice** (then cells), as elsewhere in the repo.

Indices: **OI** = (R_oddball − R_standard)/(|R_oddball|+|R_standard|); **DvI** uses the physically identical
90° grating of the equiprobable control block instead of the standard; **TPI** = tuning preference from the
control block (+1 prefers the 90° deviant orientation, −1 prefers the 0° standard).

**Input:** `data/crossscale_mechanism.parquet` — one row per cell, written by
[`crossscale_oddball_index.ipynb`](crossscale_oddball_index.ipynb) (Neuropixels 3 sessions / 3 mice;
mesoscope 23 sessions / 10 mice; SLAP2 3 original sessions). Runs offline in seconds.
**Outputs:** `data/responsiveness_matching.csv`, `figures/responsiveness_matching.{png,svg}`.

In [ ]:
#@title Install (Colab)
%pip -q install pandas pyarrow scipy matplotlib

In [ ]:
#@title Load the committed per-cell table (no streaming needed)
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import binomtest
_here=os.getcwd(); ROOT=os.path.dirname(_here) if os.path.basename(_here)=="notebooks" else _here
if not os.path.exists(os.path.join(ROOT,"data","crossscale_mechanism.parquet")):   # Colab: fetch the repo
    os.system("git clone -q https://github.com/maierav/ai_oscp_neuro.git"); ROOT="ai_oscp_neuro"
mech=pd.read_parquet(os.path.join(ROOT,"data","crossscale_mechanism.parquet"))
MODS=["ecephys","mesoscope","slap2"]; MODCOL={"ecephys":"#2c3e8c","mesoscope":"#c0392b","slap2":"#8e44ad"}
print({m:(int((mech.modality==m).sum()), int(mech[mech.modality==m].subject.nunique())) for m in MODS}, "(cells, mice)")


## Level 1 · One responsiveness rule

In [ ]:
#@title Level 1 — the same responsiveness rule for every technique
# 'resp' is asymmetric: Neuropixels admits cells significantly SUPPRESSED by the standard (Wilcoxon p<0.05,
# either sign); imaging requires p<0.05 AND a positive mean. "Matched" applies the imaging rule everywhere.
L1=[]
for mod in MODS:
    d=mech[mech.modality==mod]
    for lab,sub in [("asymmetric (current)",d[d.resp]),("matched (excitatory-only)",d[d.resp&(d.R_std>0)])]:
        L1.append(dict(level=1,modality=mod,subset=lab,n=len(sub),n_mice=sub.subject.nunique(),
                       TPI=sub.TPI.median(),OI=sub.OI.median(),DvI=sub.DvI.median() if sub.DvI.notna().any() else np.nan))
L1=pd.DataFrame(L1); print(L1.drop(columns="level").round(3).to_string(index=False))


## Level 2 · Detection floor

In [ ]:
#@title Level 2 — detection-floor test: threshold Neuropixels the way calcium imaging is thresholded
# Keep only the most strongly driven excitatory Neuropixels cells (imaging detects ~50% of cells as responsive)
# and ask whether the Neuropixels population then looks like the mesoscope one.
print("fraction responsive:",{m:round(mech[mech.modality==m].resp.mean(),2) for m in MODS})
de=mech[(mech.modality=="ecephys")&mech.resp&(mech.R_std>0)].copy(); de["rank_pct"]=de.R_std.rank(pct=True)
L2=[]
for lab,thr in [("all excitatory",0),("top 75%",0.25),("top 50%",0.5),("top 25%",0.75)]:
    s=de[de.rank_pct>thr]
    L2.append(dict(level=2,modality="ecephys",subset=lab,n=len(s),n_mice=s.subject.nunique(),TPI=s.TPI.median(),
                   sat=(s.TPI.abs()>0.9).mean(),OI=s.OI.median(),DvI=s.DvI.median()))
mr=mech[(mech.modality=="mesoscope")&mech.resp]
L2.append(dict(level=2,modality="mesoscope",subset="reference (all responsive)",n=len(mr),n_mice=mr.subject.nunique(),
               TPI=mr.TPI.median(),sat=(mr.TPI.abs()>0.9).mean(),OI=mr.OI.median(),DvI=mr.DvI.median()))
L2=pd.DataFrame(L2); print(L2.drop(columns="level").round(3).to_string(index=False))


## Level 3 · Joint balancing, mouse-level inference

In [ ]:
#@title Level 3 — joint balancing on response strength AND tuning, with mouse-level inference
# Bin responsive cells by tuning preference (6 TPI bins) x response strength (tertiles of |R_std| within technique)
# and draw the same number of cells (median bin count) from every occupied bin. Two intervals:
#   draw-only CI  = variability of the balanced draw (the original analysis; ignores between-mouse variance)
#   mouse CI      = resample MICE with replacement, then balance (the repo-wide convention; use this one)
# plus a per-mouse balanced median and an exact sign test across mice.
def _bins(d,ntpi=6,nmag=3):
    tb=np.clip(np.digitize(d.TPI.values,np.linspace(-1,1,ntpi+1))-1,0,ntpi-1)
    mb=np.clip(np.digitize(d.R_std.abs().rank(pct=True).values,np.linspace(0,1,nmag+1))-1,0,nmag-1)
    return tb*nmag+mb
def balanced_draw(d,rng,col):
    c=_bins(d); occ,cnt=np.unique(c,return_counts=True); k=max(int(np.median(cnt)),3); v=d[col].values
    pick=np.concatenate([rng.choice(np.where(c==o)[0],k,replace=cnt[i]<k) for i,o in enumerate(occ)])
    return np.nanmedian(v[pick]), k*len(occ)
def joint_balanced(mod,col,reps=1000,seed=0):
    d=mech[(mech.modality==mod)&mech.resp&mech[col].notna()&mech.TPI.notna()].reset_index(drop=True)
    rng=np.random.default_rng(seed); subs=d.subject.unique(); by={s:d[d.subject==s].reset_index(drop=True) for s in subs}
    draw=[balanced_draw(d,rng,col)[0] for _ in range(reps)]
    hier=[balanced_draw(pd.concat([by[s] for s in rng.choice(subs,len(subs),True)],ignore_index=True),rng,col)[0] for _ in range(reps)]
    pm={s:float(np.median([balanced_draw(by[s],rng,col)[0] for _ in range(200)])) for s in subs}
    npos=sum(v>0 for v in pm.values()); nz=sum(v!=0 for v in pm.values())
    return dict(level=3,modality=mod,index=col,raw=d[col].median(),balanced=float(np.median(draw)),
                draw_lo=np.percentile(draw,2.5),draw_hi=np.percentile(draw,97.5),
                mouse_lo=np.percentile(hier,2.5),mouse_hi=np.percentile(hier,97.5),
                n=balanced_draw(d,rng,col)[1],n_mice=len(subs),mice_pos=npos,mice_nonzero=nz,
                sign_p=binomtest(npos,nz,0.5).pvalue if nz else np.nan,per_mouse=pm)
L3=pd.DataFrame([joint_balanced(m,"OI") for m in MODS]+[joint_balanced(m,"DvI") for m in ["ecephys","mesoscope"]])
print(L3.drop(columns=["level","per_mouse"]).round(3).to_string(index=False))


In [ ]:
#@title Save the results table
OUT=pd.concat([L1.assign(index="—"),L2.assign(index="—"),L3.drop(columns="per_mouse").rename(columns={"index":"index"})],ignore_index=True)
OUT.to_csv(os.path.join(ROOT,"data","responsiveness_matching.csv"),index=False)
print("wrote data/responsiveness_matching.csv:",len(OUT),"rows")


## Figure

In [ ]:
#@title Figure
fig,ax=plt.subplots(1,3,figsize=(16,5.0),gridspec_kw=dict(wspace=0.34,left=0.055,right=0.985,top=0.80,bottom=0.17))
# A: Level 1
x=np.arange(3); w=0.36
cur=[L1[(L1.modality==m)&(L1.subset.str.startswith("asym"))].TPI.iat[0] for m in MODS]
mat=[L1[(L1.modality==m)&(L1.subset.str.startswith("matched"))].TPI.iat[0] for m in MODS]
ax[0].bar(x-w/2,cur,w,color="0.72",label="asymmetric rule"); ax[0].bar(x+w/2,mat,w,color=[MODCOL[m] for m in MODS],label="matched (excitatory-only)")
for xi,c,m in zip(x,cur,mat):
    ax[0].text(xi-w/2,c-0.02,f"{c:+.2f}",ha="center",va="top",fontsize=7.5,color="0.35"); ax[0].text(xi+w/2,m-0.02,f"{m:+.2f}",ha="center",va="top",fontsize=7.5,fontweight="bold")
ax[0].axhline(0,color="k",lw=0.8); ax[0].set_xticks(x); ax[0].set_xticklabels(MODS); ax[0].set_ylim(-0.85,0.08)
ax[0].set_ylabel("population tuning bias (median TPI)"); ax[0].legend(frameon=False,fontsize=8,loc="lower left")
ax[0].set_title(f"A · Same responsiveness rule for all techniques\nNeuropixels TPI {cur[0]:+.2f} → {mat[0]:+.2f}; imaging unchanged",loc="left",fontsize=9.5)
# B: Level 2
e=L2[L2.modality=="ecephys"]; xb=np.arange(len(e)); ref=L2[L2.modality=="mesoscope"].iloc[0]
ax[1].plot(xb,e.TPI,"o-",color=MODCOL["ecephys"],lw=1.8,label="Neuropixels TPI")
ax[1].plot(xb,e.OI,"s-",color="#e67e22",lw=1.8,label="Neuropixels OI")
ax[1].axhline(ref.TPI,color=MODCOL["mesoscope"],ls="--",lw=1.2,label=f"mesoscope TPI ({ref.TPI:+.2f})")
ax[1].axhline(ref.OI,color=MODCOL["mesoscope"],ls=":",lw=1.4,label=f"mesoscope OI ({ref.OI:+.2f})")
ax[1].axhline(0,color="0.6",lw=0.6)
ax[1].set_xticks(xb); ax[1].set_xticklabels(e.subset,fontsize=8); ax[1].set_ylim(-0.8,0.2)
ax[1].set_xlabel("Neuropixels cells kept (by response strength)"); ax[1].set_ylabel("median index")
ax[1].legend(frameon=False,fontsize=7.5,loc="upper right",ncol=2)
ax[1].set_title(f"B · Thresholding Neuropixels like imaging moves it toward mesoscope\n|TPI|>0.9: Neuropixels {e.sat.min():.0%}–{e.sat.max():.0%} at any floor vs mesoscope {ref.sat:.0%}",loc="left",fontsize=9.5)
# C: Level 3
o=L3[L3["index"]=="OI"].set_index("modality").loc[MODS]
ax[2].bar(x-w/2,o.raw,w,color="0.72",label="raw OI (responsive cells)")
ax[2].bar(x+w/2,o.balanced,w,color=[MODCOL[m] for m in MODS],alpha=0.85,label="joint-balanced OI")
ax[2].errorbar(x+w/2,o.balanced,yerr=[o.balanced-o.mouse_lo,o.mouse_hi-o.balanced],fmt="none",ecolor="k",capsize=4,lw=1.2)
for xi,m in zip(x,MODS):
    pm=np.array(list(o.loc[m,"per_mouse"].values())); jit=np.linspace(-0.09,0.09,len(pm))
    ax[2].scatter(xi+w/2+jit,pm,s=14,facecolor="white",edgecolor="k",lw=0.7,zorder=3,label="per-mouse balanced" if m=="ecephys" else None)
    ax[2].text(xi+w/2,max(o.loc[m,"mouse_hi"],pm.max())+0.04,f"{o.loc[m,'balanced']:+.2f}\n{int(o.loc[m,'mice_pos'])}/{int(o.loc[m,'mice_nonzero'])} mice +",
               ha="center",va="bottom",fontsize=7.5)
    r=o.loc[m,"raw"]; ax[2].text(xi-w/2,r+(0.02 if r>0 else -0.03),f"{r:+.2f}",ha="center",va="bottom" if r>0 else "top",fontsize=7.5,color="0.35")
ax[2].axhline(0,color="k",lw=0.8); ax[2].set_xticks(x); ax[2].set_xticklabels(MODS); ax[2].set_ylim(-0.62,0.9)
ax[2].set_ylabel("oddball index OI (median)"); ax[2].legend(frameon=False,fontsize=7.5,loc="upper left")
ax[2].set_title("C · Joint balance (response strength × tuning)\nerror bars: 95% CI resampling mice",loc="left",fontsize=9.5)
for a in ax:
    for sp in ["top","right"]: a.spines[sp].set_visible(False)
fig.suptitle("Visual responsiveness is an upstream confound — matching it as well as tuning brings the oddball index into agreement",fontsize=11.5)
for ext in ("png","svg"): fig.savefig(os.path.join(ROOT,"figures",f"responsiveness_matching.{ext}"),dpi=180,bbox_inches="tight")
plt.show()


## Takeaway

- **Level 1.** The asymmetric rule makes Neuropixels look less tuning-biased than a matched comparison
  allows: excitatory-only moves its median TPI from −0.06 to −0.12 (imaging populations are unchanged,
  they already use that rule). DvI barely moves (+0.34 → +0.37).
- **Level 2.** Keeping only the strongly driven Neuropixels cells pushes its OI from +0.03 to −0.10 at
  the imaging-like 50 % floor and −0.14 at 25 %, i.e. toward the mesoscope's −0.41; the tuning bias moves
  toward the mesoscope too, though not monotonically (−0.12 → −0.24 → −0.19). The saturation of tuning
  (|TPI|>0.9: 33–41 % Neuropixels at any floor vs 53 % mesoscope) is not reproduced — that part is
  calcium-specific.
- **Level 3.** Balancing response strength and tuning together gives a **positive OI** in Neuropixels
  (+0.14, mouse CI [+0.02, +0.26], 3 mice) and mesoscope (+0.21 [+0.09, +0.30], 10 mice). Mouse by
  mouse the mesoscope result is less uniform (7/10 positive, sign test p = 0.34) than the balanced
  **DvI**, which is positive in 10/10 mice (+0.19, p = 0.002).
- **SLAP2 caveat.** With no equiprobable control block, SLAP2's tuning index and oddball index come from the
  same 0°-vs-90° comparison, so balancing on one cannot separate them (balanced OI −0.12, 1/3 mice positive).